# Simple Understanding Notebook for Data Preprocessing

This notebook explains the preprocessing logic in an easy-to-read way.
It is only for understanding the flow of the project, not for production use.

We do the following in simple steps:
1. Load raw files
2. Clean values
3. Standardize IDs
4. Validate references
5. Save cleaned CSV files

In [10]:
from pathlib import Path
import pandas as pd

candidate = Path.cwd().resolve()
ROOT = candidate
for parent in [candidate, *candidate.parents]:
    if (parent / 'README.md').exists() and (parent / 'Datasets' / 'raw').exists():
        ROOT = parent
        break

RAW = ROOT / 'Datasets' / 'raw'
OUT = ROOT / 'Datasets' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

print('Root folder:', ROOT)
print('Raw folder:', RAW)
print('Output folder:', OUT)
print('Datasets exists:', (ROOT / 'Datasets').exists())


Root folder: D:\ai-multi-agent-supply-chain
Raw folder: D:\ai-multi-agent-supply-chain\Datasets\raw
Output folder: D:\ai-multi-agent-supply-chain\Datasets\processed
Datasets exists: True


## Step 1: Common helper functions

These functions are used repeatedly across the project.
- Convert dates to ISO format like `2024-01-15`
- Create `week_id` like `2024-W01`
- Create standardized IDs like `PRO-001` or `WH-001`

In [11]:
def to_iso_date(value):
    if pd.isna(value):
        return None
    dt = pd.to_datetime(value, errors='coerce')
    return dt.strftime('%Y-%m-%d') if pd.notna(dt) else None

def week_id_from_date(value):
    if pd.isna(value):
        return None
    dt = pd.to_datetime(value, errors='coerce')
    if pd.isna(dt):
        return None
    iso = dt.isocalendar()
    return f"{iso.year}-W{iso.week:02d}"

def standard_id(prefix, n, width=3):
    return f'{prefix}-{n:0{width}d}'

print(to_iso_date('2024-01-15'))
print(week_id_from_date('2024-01-15'))
print(standard_id('PRO', 1))

2024-01-15
2024-W03
PRO-001


## Step 2: Clean products dataset

We read the raw product master file and keep only the important columns.
Then we remove rows that are missing important business data, such as product name, brand, or selling price.

In [12]:
products_raw = pd.read_csv(RAW / 'master' / 'products.csv')
print('Raw products rows before cleaning:', len(products_raw))
products_raw.head(5)

Raw products rows before cleaning: 200


,product_id,category_code,category_name,product_name,brand,quality_level,unit_type,unit_size_1,cp_1_rs,sp_1_rs,...,sp_2_rs,unit_size_3,cp_3_rs,sp_3_rs,unit_size_4,cp_4_rs,sp_4_rs,unit_size_5,cp_5_rs,sp_5_rs
0,ICE-001,ICE,Ice Cream & Frozen Desserts,Amul Vanilla Ice Cream,Amul,Standard,ml,80 ml,36,41,...,60,250 ml,82,94,500 ml,120,138,1 L,218,251
1,ICE-002,ICE,Ice Cream & Frozen Desserts,Amul Chocolate Ice Cream,Amul,Standard,ml,80 ml,36,41,...,60,250 ml,82,94,500 ml,120,138,1 L,218,251
2,ICE-003,ICE,Ice Cream & Frozen Desserts,Kwality Wall's Cornetto,Kwality Wall's,Standard,ml,80 ml,48,55,...,79,250 ml,109,125,500 ml,160,184,1 L,291,335
3,ICE-004,ICE,Ice Cream & Frozen Desserts,Kwality Wall's Magnum Classic,Kwality Wall's,Premium,ml,80 ml,78,90,...,129,250 ml,177,204,500 ml,260,299,1 L,473,544
4,ICE-005,ICE,Ice Cream & Frozen Desserts,Havmor Butterscotch Ice Cream,Havmor,Standard,ml,80 ml,51,59,...,84,250 ml,116,133,500 ml,170,195,1 L,309,355


In [13]:
products = products_raw[[
    'product_id', 'category_code', 'category_name', 'product_name', 'brand',
    'quality_level', 'unit_type', 'unit_size_1', 'cp_1_rs', 'sp_1_rs'
]].copy()

products = products.dropna(subset=['product_name', 'brand', 'sp_1_rs']).reset_index(drop=True)
products['product_id'] = [standard_id('PRO', i) for i in range(1, len(products) + 1)]

products = products.rename(columns={
    'unit_size_1': 'unit_size',
    'cp_1_rs': 'cost_price',
    'sp_1_rs': 'selling_price'
})

products = products[[
    'product_id', 'category_code', 'category_name', 'product_name', 'brand',
    'quality_level', 'unit_type', 'unit_size', 'cost_price', 'selling_price'
]].reset_index(drop=True)

print('Products rows after cleaning:', len(products))
products.head(5)

Products rows after cleaning: 200


,product_id,category_code,category_name,product_name,brand,quality_level,unit_type,unit_size,cost_price,selling_price
0,PRO-001,ICE,Ice Cream & Frozen Desserts,Amul Vanilla Ice Cream,Amul,Standard,ml,80 ml,36,41
1,PRO-002,ICE,Ice Cream & Frozen Desserts,Amul Chocolate Ice Cream,Amul,Standard,ml,80 ml,36,41
2,PRO-003,ICE,Ice Cream & Frozen Desserts,Kwality Wall's Cornetto,Kwality Wall's,Standard,ml,80 ml,48,55
3,PRO-004,ICE,Ice Cream & Frozen Desserts,Kwality Wall's Magnum Classic,Kwality Wall's,Premium,ml,80 ml,78,90
4,PRO-005,ICE,Ice Cream & Frozen Desserts,Havmor Butterscotch Ice Cream,Havmor,Standard,ml,80 ml,51,59


## Step 3: Clean locations dataset

This dataset holds region information. We remove duplicate rows and keep only clean location references.

In [14]:
locations_raw = pd.read_csv(RAW / 'master' / 'locations.csv.csv')
print('Raw locations rows before cleaning:', len(locations_raw))
locations_raw.head(5)

Raw locations rows before cleaning: 40


,location_id,region_of_kolkata
0,KOL-LOC-001,Lake Gardens
1,KOL-LOC-002,Jadavpur
2,KOL-LOC-003,Dhakuria
3,KOL-LOC-004,Gariahat
4,KOL-LOC-005,Ballygunge


In [15]:
locations = locations_raw[[
    'location_id', 'region_of_kolkata'
]].drop_duplicates().dropna().reset_index(drop=True)

locations = locations.rename(columns={'region_of_kolkata': 'region_name'})
print('Locations rows after cleaning:', len(locations))
locations.head(5)

Locations rows after cleaning: 40


,location_id,region_name
0,KOL-LOC-001,Lake Gardens
1,KOL-LOC-002,Jadavpur
2,KOL-LOC-003,Dhakuria
3,KOL-LOC-004,Gariahat
4,KOL-LOC-005,Ballygunge


## Step 4: Clean warehouses dataset

We read the warehouse workbook, choose the correct sheet, and assign a new warehouse ID.

In [16]:
def choose_excel_sheet(path, preferred):
    xls = pd.ExcelFile(path, engine='openpyxl')
    for name in preferred:
        if name in xls.sheet_names:
            return name
    return xls.sheet_names[0]

warehouse_path = RAW / 'warehouse' / 'warehouses.xlsx'
sheet_name = choose_excel_sheet(warehouse_path, ['warehouse'])
sheet_name

'warehouse'

In [17]:
warehouse_raw = pd.read_excel(warehouse_path, sheet_name=sheet_name, engine='openpyxl')
print('Raw warehouse rows before cleaning:', len(warehouse_raw))
warehouse_raw = warehouse_raw[[
    'warehouse_id', 'warehouse_name', 'area', 'city', 'capacity_units', 'daily_dispatch_capacity_units'
]].dropna(subset=['warehouse_name']).reset_index(drop=True)

warehouse_raw['warehouse_id'] = [standard_id('WH', i) for i in range(1, len(warehouse_raw) + 1)]
warehouse_raw['location_id'] = [locations['location_id'].iloc[i % len(locations)] for i in range(len(warehouse_raw))]

warehouses = warehouse_raw[[
    'warehouse_id', 'warehouse_name', 'area', 'city', 'capacity_units', 'daily_dispatch_capacity_units', 'location_id'
]].reset_index(drop=True)

print('Warehouse rows after cleaning:', len(warehouses))
warehouses.head(5)

Raw warehouse rows before cleaning: 15
Warehouse rows after cleaning: 15


,warehouse_id,warehouse_name,area,city,capacity_units,daily_dispatch_capacity_units,location_id
0,WH-001,South City Mall Warehouse,Jadavpur,Kolkata,20000,7000,KOL-LOC-001
1,WH-002,Quest Mall Warehouse,Park Circus,Kolkata,16000,5600,KOL-LOC-002
2,WH-003,Acropolis Mall Warehouse,Kasba,Kolkata,18000,6300,KOL-LOC-003
3,WH-004,Mani Square Warehouse,Kankurgachi,Kolkata,18000,6300,KOL-LOC-004
4,WH-005,City Centre I Warehouse,Salt Lake,Kolkata,20000,7000,KOL-LOC-005


## Step 5: Clean suppliers dataset

We combine supplier master data and product catalog data, then assign standard supplier IDs.

In [18]:
supplier_path = RAW / 'supplier' / 'supplier_inventory.xlsx'
supplier_master = pd.read_excel(supplier_path, sheet_name=choose_excel_sheet(supplier_path, ['Supplier_Master']), engine='openpyxl')
catalog = pd.read_excel(supplier_path, sheet_name=choose_excel_sheet(supplier_path, ['Supplier_Product_Catalog']), engine='openpyxl')

print('Raw supplier master rows before cleaning:', len(supplier_master))
supplier_master.head(5)

Raw supplier master rows before cleaning: 200


,supplier_id,supplier_name,supplier_type_id,supplier_type_name,location_id,location_name,city,supplier_latitude,supplier_longitude,product_category_scope,minimum_order_qty_units,max_order_qty_units,max_ship_qty_at_once_units,supplier_storage_capacity_units,vehicle_type,vehicle_count,vehicle_load_capacity_units,lead_time_days,location_note
0,SUP-KOL-001-01,Lake Gardens Fresh Produce & Chilled Market,SUP-T01,Fresh Produce & Chilled,KOL-LOC-001,Lake Gardens,Kolkata,22.5051,88.3542,Fresh Fruits; Fresh Vegetables; Dairy & Milk P...,90,1900,750,2300,Refrigerated Mini Truck,4,34,1,Approximate planning coordinate near area center
1,SUP-KOL-001-02,Lake Gardens Staples & Cooking Depot,SUP-T02,Staples & Cooking,KOL-LOC-001,Lake Gardens,Kolkata,22.5078,88.3544,"Rice & Staples; Pulses & Dal; Flour, Atta & Ba...",110,2300,1050,3100,Mini Truck,5,30,1,Approximate planning coordinate near area center
2,SUP-KOL-001-03,Lake Gardens Packaged Foods & Bakery Distribut...,SUP-T03,Packaged Foods & Bakery,KOL-LOC-001,Lake Gardens,Kolkata,22.5055,88.3569,Biscuits & Cookies; Bread & Bakery; Chips & Na...,70,1700,800,2500,Tempo/LCV,4,28,1,Approximate planning coordinate near area center
3,SUP-KOL-001-04,Lake Gardens Beverages Supply Point,SUP-T04,Beverages,KOL-LOC-001,Lake Gardens,Kolkata,22.5076,88.3568,Soft Drinks & Beverages; Tea & Coffee,90,1900,950,1900,Van,4,26,1,Approximate planning coordinate near area center
4,SUP-KOL-001-05,"Lake Gardens Home, Personal & Family Wholesale...",SUP-T05,"Home, Personal & Family",KOL-LOC-001,Lake Gardens,Kolkata,22.5065,88.3554,Baby Care & Family Care; Household Essentials;...,60,1500,650,1900,Mini Truck,4,29,1,Approximate planning coordinate near area center


In [19]:
product_map = {old: new for old, new in zip(pd.read_csv(RAW / 'master' / 'products.csv')['product_id'].tolist(), products['product_id'].tolist())}
supplier_product_map = catalog[[
    'supplier_id', 'product_supplied_id'
]].dropna().drop_duplicates().groupby('supplier_id')['product_supplied_id'].first().to_dict()

suppliers = supplier_master[[
    'supplier_id', 'supplier_name', 'supplier_type_name', 'location_id'
]].copy()
suppliers['product_id_raw'] = suppliers['supplier_id'].map(supplier_product_map)
suppliers['product_id'] = suppliers['product_id_raw'].map(product_map).fillna(products['product_id'].iloc[0])
suppliers['supplier_id'] = [standard_id('SUP', i) for i in range(1, len(suppliers) + 1)]

suppliers = suppliers[[
    'supplier_id', 'supplier_name', 'supplier_type_name', 'location_id', 'product_id'
]].rename(columns={'supplier_type_name': 'supplier_type'})

suppliers = suppliers.dropna(subset=['supplier_name', 'location_id', 'product_id']).reset_index(drop=True)
print('Supplier rows after cleaning:', len(suppliers))
suppliers.head(5)

Supplier rows after cleaning: 200


,supplier_id,supplier_name,supplier_type,location_id,product_id
0,SUP-001,Lake Gardens Fresh Produce & Chilled Market,Fresh Produce & Chilled,KOL-LOC-001,PRO-001
1,SUP-002,Lake Gardens Staples & Cooking Depot,Staples & Cooking,KOL-LOC-001,PRO-061
2,SUP-003,Lake Gardens Packaged Foods & Bakery Distribut...,Packaged Foods & Bakery,KOL-LOC-001,PRO-011
3,SUP-004,Lake Gardens Beverages Supply Point,Beverages,KOL-LOC-001,PRO-031
4,SUP-005,"Lake Gardens Home, Personal & Family Wholesale...","Home, Personal & Family",KOL-LOC-001,PRO-141


## Step 6: Clean inventory dataset

Inventory is a bridge table between warehouse, product, and shelf.
We combine product/shelf assignment with stock position data and compute safety stock.

In [20]:
inventory_path = RAW / 'inventory' / 'inventory_stock.xlsx'
assignment = pd.read_excel(inventory_path, sheet_name='Product_Shelf_Assignment', engine='openpyxl')
position = pd.read_excel(inventory_path, sheet_name='Inventory_Position', engine='openpyxl')

print('Raw assignment rows before cleaning:', len(assignment))
assignment.head(5)

Raw assignment rows before cleaning: 3000


,warehouse_id,product_id,shelf_id,bin_id,zone_id,rack_id,shelf_level,slot_no,slot_capacity_units
0,WH-KOL-001,ICE-001,WH-KOL-001-Z01-R01-S01,WH-KOL-001-Z01-R01-S01-B01,Z01,R01,S01,1,200
1,WH-KOL-001,ICE-002,WH-KOL-001-Z01-R01-S01,WH-KOL-001-Z01-R01-S01-B02,Z01,R01,S01,2,200
2,WH-KOL-001,ICE-003,WH-KOL-001-Z01-R01-S01,WH-KOL-001-Z01-R01-S01-B03,Z01,R01,S01,3,200
3,WH-KOL-001,ICE-004,WH-KOL-001-Z01-R01-S01,WH-KOL-001-Z01-R01-S01-B04,Z01,R01,S01,4,200
4,WH-KOL-001,ICE-005,WH-KOL-001-Z01-R01-S01,WH-KOL-001-Z01-R01-S01-B05,Z01,R01,S01,5,200


In [21]:
product_map = {old: new for old, new in zip(pd.read_csv(RAW / 'master' / 'products.csv')['product_id'].tolist(), products['product_id'].tolist())}
warehouse_map = {old: new for old, new in zip(pd.read_excel(RAW / 'warehouse' / 'warehouses.xlsx', sheet_name='warehouse', engine='openpyxl')['warehouse_id'].tolist(), warehouses['warehouse_id'].tolist())}

inventory = assignment[['warehouse_id', 'product_id', 'shelf_id']].copy()
position = position[['warehouse_id', 'product_id', 'available_stock_units']].rename(columns={'available_stock_units': 'available_stock'})
inventory = inventory.merge(position, on=['warehouse_id', 'product_id'], how='left')

inventory['available_stock'] = pd.to_numeric(inventory['available_stock'], errors='coerce').fillna(0)
inventory['safety_stock'] = (inventory['available_stock'] * 0.15).round().astype(int)
inventory = inventory.drop_duplicates(subset=['warehouse_id', 'product_id', 'shelf_id']).reset_index(drop=True)

inventory['warehouse_id'] = inventory['warehouse_id'].map(warehouse_map)
inventory['product_id'] = inventory['product_id'].map(product_map)
inventory['inventory_id'] = [standard_id('INV', i) for i in range(1, len(inventory) + 1)]

inventory = inventory[[
    'inventory_id', 'warehouse_id', 'product_id', 'shelf_id', 'available_stock', 'safety_stock'
]].reset_index(drop=True)
print('Inventory rows after cleaning:', len(inventory))
inventory.head(5)

Inventory rows after cleaning: 3000


,inventory_id,warehouse_id,product_id,shelf_id,available_stock,safety_stock
0,INV-001,WH-001,PRO-001,WH-KOL-001-Z01-R01-S01,77,12
1,INV-002,WH-001,PRO-002,WH-KOL-001-Z01-R01-S01,95,14
2,INV-003,WH-001,PRO-003,WH-KOL-001-Z01-R01-S01,63,9
3,INV-004,WH-001,PRO-004,WH-KOL-001-Z01-R01-S01,76,11
4,INV-005,WH-001,PRO-005,WH-KOL-001-Z01-R01-S01,72,11


## Step 7: Clean weather dataset

We convert weekly date strings into ISO format and assign week IDs.

In [22]:
weather_raw = pd.read_csv(RAW / 'external' / 'weather_weekly.csv')
print('Raw weather rows before cleaning:', len(weather_raw))
weather_raw.head(5)

Raw weather rows before cleaning: 105


,week_start_date,week_end_date,year,season,temperature_mean_c,temperature_max_c,temperature_min_c,rainfall_mm,humidity_pct,weather_condition
0,2024-01-01,2024-01-07,2024,Winter,18.6,22.3,16.1,12.9,56.8,Clear/Partly Cloudy
1,2024-01-08,2024-01-14,2024,Winter,19.4,20.8,17.3,4.5,62.3,Clear/Partly Cloudy
2,2024-01-15,2024-01-21,2024,Winter,19.0,21.2,16.3,2.1,65.3,Clear/Partly Cloudy
3,2024-01-22,2024-01-28,2024,Winter,19.4,22.1,16.9,5.0,56.1,Clear/Partly Cloudy
4,2024-01-29,2024-02-04,2024,Winter,21.0,24.4,16.6,3.5,56.5,Clear/Partly Cloudy


In [23]:
weather = weather_raw[[
    'week_start_date', 'week_end_date', 'season', 'temperature_mean_c',
    'rainfall_mm', 'humidity_pct', 'weather_condition'
]].copy()

weather['week_start_date'] = weather['week_start_date'].map(to_iso_date)
weather['week_end_date'] = weather['week_end_date'].map(to_iso_date)
weather['week_id'] = weather['week_start_date'].map(week_id_from_date)
weather['weather_id'] = [standard_id('WEA', i) for i in range(1, len(weather) + 1)]

weather = weather[[
    'weather_id', 'week_id', 'week_start_date', 'week_end_date', 'season',
    'temperature_mean_c', 'rainfall_mm', 'humidity_pct', 'weather_condition'
]].dropna(subset=['week_id']).reset_index(drop=True)
print('Weather rows after cleaning:', len(weather))
weather.head(5)

Weather rows after cleaning: 105


,weather_id,week_id,week_start_date,week_end_date,season,temperature_mean_c,rainfall_mm,humidity_pct,weather_condition
0,WEA-001,2024-W01,2024-01-01,2024-01-07,Winter,18.6,12.9,56.8,Clear/Partly Cloudy
1,WEA-002,2024-W02,2024-01-08,2024-01-14,Winter,19.4,4.5,62.3,Clear/Partly Cloudy
2,WEA-003,2024-W03,2024-01-15,2024-01-21,Winter,19.0,2.1,65.3,Clear/Partly Cloudy
3,WEA-004,2024-W04,2024-01-22,2024-01-28,Winter,19.4,5.0,56.1,Clear/Partly Cloudy
4,WEA-005,2024-W05,2024-01-29,2024-02-04,Winter,21.0,3.5,56.5,Clear/Partly Cloudy


## Step 8: Clean festival dataset

Each festival event is mapped to the week it happened in.

In [24]:
festival_raw = pd.read_csv(RAW / 'external' / 'festival_calendar.csv')
print('Raw festival rows before cleaning:', len(festival_raw))
festival_raw.head(5)

Raw festival rows before cleaning: 36


,festival_event,event_date,year
0,Saraswati Puja,2024-02-14,2024
1,Eid-Ul-Fitr,2024-04-11,2024
2,Poila Boishakh / Bengali New Year,2024-04-14,2024
3,Rath Yatra,2024-07-07,2024
4,Janmashtami,2024-08-26,2024


In [25]:
festivals = festival_raw[['festival_event', 'event_date', 'year']].copy()
festivals['event_date'] = festivals['event_date'].map(to_iso_date)
festivals['week_id'] = festivals['event_date'].map(week_id_from_date)
festivals['festival_id'] = [standard_id('FES', i) for i in range(1, len(festivals) + 1)]

festivals = festivals[[
    'festival_id', 'festival_event', 'event_date', 'year', 'week_id'
]].dropna(subset=['festival_event', 'event_date']).reset_index(drop=True)
print('Festival rows after cleaning:', len(festivals))
festivals.head(5)

Festival rows after cleaning: 36


,festival_id,festival_event,event_date,year,week_id
0,FES-001,Saraswati Puja,2024-02-14,2024,2024-W07
1,FES-002,Eid-Ul-Fitr,2024-04-11,2024,2024-W15
2,FES-003,Poila Boishakh / Bengali New Year,2024-04-14,2024,2024-W15
3,FES-004,Rath Yatra,2024-07-07,2024,2024-W27
4,FES-005,Janmashtami,2024-08-26,2024,2024-W35


## Step 9: Clean sales dataset

Sales is the main transactional dataset. We standardize product IDs, add `week_id`, and calculate revenue.

In [26]:
sales_path = RAW / 'demand' / 'Demand of last 2 years.xlsx'
sales_raw = pd.read_excel(sales_path, sheet_name='demand_training_data', engine='openpyxl')
print('Raw sales rows before cleaning:', len(sales_raw))
sales_raw.head(5)

Raw sales rows before cleaning: 27800


,demand_id,week_start_date,week_end_date,year,week_number,location_id,region_of_kolkata,demand_rank,product_id,product_name,...,region_product_week_key,assortment_size,rank_scope,rank_basis,selling_unit_workbook,selling_unit_sheet,selling_unit_source_row,selling_unit_lookup_hint,reference_layer,anchor_status
0,DEM-0000001,2024-01-01 00:00:00,2024-01-07 00:00:00,2024,1,KOL-LOC-001,Lake Gardens,1,DAI-005,Mother Dairy Curd,...,KOL-LOC-001_20240101_DAI-005,200,region-week over all 200 products,units_sold DESC; sales_value_rs DESC; product_...,Regional_Selling_Unit_40Regions_200Products_13...,W001_2024-01-01,3,'W001_2024-01-01'!A3,regional_selling_unit_reference,observed_top5_anchor
1,DEM-0000002,2024-01-01 00:00:00,2024-01-07 00:00:00,2024,1,KOL-LOC-001,Lake Gardens,2,TEA-009,Bru Instant Coffee,...,KOL-LOC-001_20240101_TEA-009,200,region-week over all 200 products,units_sold DESC; sales_value_rs DESC; product_...,Regional_Selling_Unit_40Regions_200Products_13...,W001_2024-01-01,4,'W001_2024-01-01'!A4,regional_selling_unit_reference,observed_top5_anchor
2,DEM-0000003,2024-01-01 00:00:00,2024-01-07 00:00:00,2024,1,KOL-LOC-001,Lake Gardens,3,TEA-002,Tata Tea Premium,...,KOL-LOC-001_20240101_TEA-002,200,region-week over all 200 products,units_sold DESC; sales_value_rs DESC; product_...,Regional_Selling_Unit_40Regions_200Products_13...,W001_2024-01-01,5,'W001_2024-01-01'!A5,regional_selling_unit_reference,observed_top5_anchor
3,DEM-0000004,2024-01-01 00:00:00,2024-01-07 00:00:00,2024,1,KOL-LOC-001,Lake Gardens,4,BEV-007,Frooti Mango,...,KOL-LOC-001_20240101_BEV-007,200,region-week over all 200 products,units_sold DESC; sales_value_rs DESC; product_...,Regional_Selling_Unit_40Regions_200Products_13...,W001_2024-01-01,6,'W001_2024-01-01'!A6,regional_selling_unit_reference,observed_top5_anchor
4,DEM-0000005,2024-01-01 00:00:00,2024-01-07 00:00:00,2024,1,KOL-LOC-001,Lake Gardens,5,INS-008,Kellogg's Corn Flakes,...,KOL-LOC-001_20240101_INS-008,200,region-week over all 200 products,units_sold DESC; sales_value_rs DESC; product_...,Regional_Selling_Unit_40Regions_200Products_13...,W001_2024-01-01,7,'W001_2024-01-01'!A7,regional_selling_unit_reference,observed_top5_anchor


In [27]:
sales = sales_raw[[
    'week_start_date', 'week_end_date', 'location_id', 'product_id',
    'units_sold', 'avg_selling_price_rs'
]].copy()

product_map = {old: new for old, new in zip(pd.read_csv(RAW / 'master' / 'products.csv')['product_id'].tolist(), products['product_id'].tolist())}
sales['week_start_date'] = sales['week_start_date'].map(to_iso_date)
sales['week_end_date'] = sales['week_end_date'].map(to_iso_date)
sales['week_id'] = sales['week_start_date'].map(week_id_from_date)
sales['product_id'] = sales['product_id'].map(product_map)
sales['units_sold'] = pd.to_numeric(sales['units_sold'], errors='coerce').fillna(0)
sales['avg_selling_price_rs'] = pd.to_numeric(sales['avg_selling_price_rs'], errors='coerce').fillna(0)
sales['revenue_rs'] = sales['units_sold'] * sales['avg_selling_price_rs']
sales['sale_id'] = [f'SALE-{i:06d}' for i in range(1, len(sales) + 1)]

sales = sales[[
    'sale_id', 'week_id', 'week_start_date', 'week_end_date', 'location_id', 'product_id',
    'units_sold', 'avg_selling_price_rs', 'revenue_rs'
]].reset_index(drop=True)
print('Sales rows after cleaning:', len(sales))
sales.head(5)

Sales rows after cleaning: 27800


,sale_id,week_id,week_start_date,week_end_date,location_id,product_id,units_sold,avg_selling_price_rs,revenue_rs
0,SALE-000001,2024-W01,2024-01-01,2024-01-07,KOL-LOC-001,PRO-045,146,52.79,7707.34
1,SALE-000002,2024-W01,2024-01-01,2024-01-07,KOL-LOC-001,PRO-139,98,273.60,26812.80
2,SALE-000003,2024-W01,2024-01-01,2024-01-07,KOL-LOC-001,PRO-132,94,293.38,27577.72
3,SALE-000004,2024-W01,2024-01-01,2024-01-07,KOL-LOC-001,PRO-037,86,38.36,3298.96
4,SALE-000005,2024-W01,2024-01-01,2024-01-07,KOL-LOC-001,PRO-188,83,78.95,6552.85


## Step 10: Validation rules

We check if there are duplicate rows, missing values, or invalid references.
This is important because downstream agents need clean data.

In [28]:
def validate(df, key_col):
    total = len(df)
    duplicates = int(df.duplicated(subset=[key_col]).sum())
    missing = int(df.isna().sum().sum())
    return {
        'total_records': total,
        'duplicate_records': duplicates,
        'missing_values': missing
    }

print('Products validation summary:', validate(products, 'product_id'))
print('Locations validation summary:', validate(locations, 'location_id'))
print('Sales validation summary:', validate(sales, 'sale_id'))

Products validation summary: {'total_records': 200, 'duplicate_records': 0, 'missing_values': 0}
Locations validation summary: {'total_records': 40, 'duplicate_records': 0, 'missing_values': 0}
Sales validation summary: {'total_records': 27800, 'duplicate_records': 0, 'missing_values': 0}


## Step 11: Save processed outputs

Once cleaned, we save each dataset to the processed folder so downstream work can use them.

In [29]:
products.to_csv(OUT / 'products.csv', index=False)
locations.to_csv(OUT / 'locations.csv', index=False)
warehouses.to_csv(OUT / 'warehouses.csv', index=False)
suppliers.to_csv(OUT / 'suppliers.csv', index=False)
inventory.to_csv(OUT / 'inventory.csv', index=False)
sales.to_csv(OUT / 'sales.csv', index=False)
weather.to_csv(OUT / 'weather.csv', index=False)
festivals.to_csv(OUT / 'festivals.csv', index=False)

print('All processed files saved successfully!')
print('Saved files:', sorted(p.name for p in OUT.iterdir() if p.suffix == '.csv'))

All processed files saved successfully!
Saved files: ['festivals.csv', 'inventory.csv', 'locations.csv', 'products.csv', 'sales.csv', 'suppliers.csv', 'warehouses.csv', 'weather.csv']


## Final note

This notebook is created only to make the preprocessing flow easier to understand.
The actual production script in the project follows the same logic but is more structured and validation-heavy.